# Species comparison
## Setup human regions
### Author: Martin Loza
### Date: 26/01/09

We will set fixed regions around human TSS of lncRNA-TF gene pairs for liftover to other species

In [35]:
# Change R language to English
Sys.setenv(LANGUAGE = "en")

# Init
suppressPackageStartupMessages({
    library(dplyr)
    library(stringr)
    library(ggplot2)
    library(patchwork)
    library(dplyr)
})

# Local variables 
seed = 777
date = "260109"

# Define colors for strand plots
red = "#E41A1C"
blue = "#377EB8"
# Define colors for gene types
green = "#4DAF4A"
purple = "#984EA3"
text_size = 18
width = 18.6
dot_size = 4
line_size = 1.5
dpi = 300

in_dir = "/mnt/c/Users/Marti/Documents/Projects/lncRNA_TF_pairs_analysis/Data/Annotated_ncRNA_PCG_pairs/"
ensembl_raw_annotation_dir = "/mnt/c/Users/Marti/Documents/Projects/lncRNA_TF_pairs_analysis/Data/ENSEMBL/selected/"
out_dir = "/mnt/c/Users/Marti/Documents/Projects/lncRNA_TF_pairs_analysis/04_Figure_window_size_and_GO/Results/"
# Local Functions



### Load and setup the data

In [36]:
# Load the setup transcripts data
# We have different species, so let's create a list to store the data
data_list = list()

# Search for the available files
files <- list.files(in_dir)

# Select files related to the species of interest
sel_species <- c("human")
file_human <- files[str_replace(files, "_.*", "") %in% sel_species]

# Load the data for each species
human_data<- read.table(file.path(in_dir, file_human), sep = "\t", header = TRUE, 
                                            stringsAsFactors = FALSE, quote = "", 
                                            comment.char = "", fill = TRUE, row.names = NULL)



head(human_data, 3)

,chromosome,ncRNA_id,ncrna_tss,ncrna_gene_name,ncrna_strand,gene_biotype,pcg_id,pcg_gene_name,pcg_tss,dna_distance,strand_distance,Family,is_TF
,<chr>,<chr>,<int>,<chr>,<int>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<chr>,<lgl>
1,19,ENST00000221567,54532791,,1,lncRNA,ENST00000590333,BRSK1,55282072,749281,749281,NA,FALSE
2,19,ENST00000221567,54532791,,1,lncRNA,ENST00000635964,C19orf85,55464751,931960,931960,NA,FALSE
3,19,ENST00000221567,54532791,,1,lncRNA,ENST00000346968,CACNG6,53992878,-539913,-539913,NA,FALSE


In [37]:
unique(human_data$chromosome)

[1] "19" "12" "14" "1"  "7"  "20" "2"  "11" "22" "Y"  "16" "17" "X"  "3"  "6" 
[16] "10" "21" "9"  "5"  "8"  "15" "4"  "18" "13"

Load the original ENSEMBL annotation to recover the pcg strand

In [38]:
# Load the raw-selected transcripts data
# We have different species, so let's create a list to store the data
data_list_raw = list()

# Search for the available files
files <- list.files(ensembl_raw_annotation_dir)

# Let's focus on human
files <- files[str_detect(files, "human")]

# Load the data for each species
for (file in files) {
    # Remove the underscore and everything after it to get the species names
    species_name <- str_replace(file, "_.*", "")
    data_list_raw[[species_name]] <- read.table(file.path(ensembl_raw_annotation_dir, file), sep = "\t", header = TRUE, 
                                            stringsAsFactors = FALSE, quote = "", 
                                            comment.char = "", fill = TRUE, row.names = NULL)
}

# We need to map start, end and strand information from the raw data to the selected data
sel_cols <- c('Transcript.stable.ID','Strand', 'Gene.stable.ID')
data_list_raw <- lapply(data_list_raw, function(df) df[, sel_cols])
head(data_list_raw[["human"]], 3)

,Transcript.stable.ID,Strand,Gene.stable.ID
,<chr>,<int>,<chr>
1,ENST00000456328,1,ENSG00000290825
2,ENST00000832823,1,ENSG00000290825
3,ENST00000832824,1,ENSG00000290825


Map the missing information to the human data

In [39]:
# Map the missing information
human_data <- human_data %>%
    # map the PCG information
    left_join(
        data_list_raw[["human"]],
        by = c("pcg_id" = "Transcript.stable.ID") 
    ) %>%
    dplyr::rename(
        pcg_strand = Strand,
        pcg_gene_id = Gene.stable.ID
    )
head(human_data, 3)

,chromosome,ncRNA_id,ncrna_tss,ncrna_gene_name,ncrna_strand,gene_biotype,pcg_id,pcg_gene_name,pcg_tss,dna_distance,strand_distance,Family,is_TF,pcg_strand,pcg_gene_id
,<chr>,<chr>,<int>,<chr>,<int>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<chr>,<lgl>,<int>,<chr>
1,19,ENST00000221567,54532791,,1,lncRNA,ENST00000590333,BRSK1,55282072,749281,749281,NA,FALSE,1,ENSG00000160469
2,19,ENST00000221567,54532791,,1,lncRNA,ENST00000635964,C19orf85,55464751,931960,931960,NA,FALSE,-1,ENSG00000283567
3,19,ENST00000221567,54532791,,1,lncRNA,ENST00000346968,CACNG6,53992878,-539913,-539913,NA,FALSE,1,ENSG00000130433


In [40]:
rm(data_list_raw)
gc()

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1479358,79.1,4535066,242.2,4882075,260.8
Vcells,55488527,423.4,177119058,1351.4,239938019,1830.6


In [41]:
# Let's annotate the transcription direction sense
human_data <-human_data %>%
    mutate(
        transcription_type = case_when(
            ncrna_strand == pcg_strand ~ "Sense",
            ncrna_strand != pcg_strand ~ "Antisense",
            TRUE ~ "Unknown"
        )
    )
table(human_data$transcription_type)


Antisense     Sense 
  2239091   2111056 

Let's focus only in lncRNAs

In [42]:
# Select only observations related to lncRNA
data_human_selected <- human_data %>%
    filter(gene_biotype == "lncRNA")
table(data_human_selected$gene_biotype)


 lncRNA 
4155667 

Let's select the genes within the selected 10kb distance

In [43]:
# Add absolute distance column
data_human_selected <- data_human_selected %>%
        mutate(abs_strand_distance = abs(strand_distance))

# Define the window sizes for downstream analysis
window_size <- 10000  # 10 kb

# Select only pairs within the defined window sizes
data_human_selected <- data_human_selected %>%
        filter(abs_strand_distance <= window_size)

cat("Number of lncRNA-PCG pairs within", window_size, "bp window in human:\n")
nrow(data_human_selected)

Number of lncRNA-PCG pairs within 10000 bp window in human:


[1] 105579

Select gene pairs related to TF

In [44]:
# Select gene pairs where the PCG is a TF
data_human_selected <- data_human_selected %>%
    filter(is_TF == TRUE)
cat("Number of lncRNA-TF pairs within", window_size, "bp window in human:\n")
nrow(data_human_selected)
cat("Number of unique lncRNA transcripts: ", length(unique(data_human_selected$ncRNA_id)), "\n")
cat("Number of unique TF genes: ", length(unique(data_human_selected$pcg_id)), "\n")

Number of lncRNA-TF pairs within 10000 bp window in human:


[1] 55454

Number of unique lncRNA transcripts:  10640 
Number of unique TF genes:  1918 


In [45]:
head(data_human_selected, 3)

,chromosome,ncRNA_id,ncrna_tss,ncrna_gene_name,ncrna_strand,gene_biotype,pcg_id,pcg_gene_name,pcg_tss,dna_distance,strand_distance,Family,is_TF,pcg_strand,pcg_gene_id,transcription_type,abs_strand_distance
,<chr>,<chr>,<int>,<chr>,<int>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<chr>,<lgl>,<int>,<chr>,<chr>,<int>
1,11,ENST00000244906,61757671,MYRF-AS1,-1,lncRNA,ENST00000265460,MYRF,61755389,-2282,2282,NDT80_PhoG,TRUE,1,ENSG00000124920,Antisense,2282
2,17,ENST00000285274,18682228,ZNF286B,-1,lncRNA,ENST00000454745,FOXO3B,18682262,34,-34,Fork_head,TRUE,-1,ENSG00000240445,Sense,34
3,5,ENST00000297163,136134890,SMAD5-AS1,-1,lncRNA,ENST00000506223,SMAD5,136133823,-1067,1067,MH1,TRUE,1,ENSG00000113658,Antisense,1067


Let's save the selected gene pairs as supplementary table

In [46]:
# table file name
pairs_folder = "/mnt/c/Users/Marti/Documents/Projects/lncRNA_TF_pairs_analysis/Data/Supplementary/"
pairs_file = paste0(pairs_folder, "Supplementary_Table_lncRNA_TF_pairs_10kb_", date, ".tsv")
# Save the lncRNA-TF pairs file
write.table(data_human_selected, file = pairs_file, sep = "\t", 
            quote = FALSE, row.names = FALSE, col.names = TRUE)

Let's setup fixed regions around transcripts TSS for liftover comparison across species

In [ ]:
fix_width = 1000
# Get the human lncRNA transcripts and setup a fixed region around TSS
human_lncRNA_transcripts <- data_human_selected %>%
    select(ncRNA_id, chromosome, ncrna_tss) %>%
    mutate(region_start = as.integer(floor(ncrna_tss - fix_width / 2)),
           region_end = as.integer(floor(ncrna_tss + fix_width / 2))) %>%
    distinct() %>%
    select(chromosome, region_start, region_end, ncRNA_id) %>%
    filter(chromosome %in% c(1:22, "X")) %>%
    mutate(chromosome = paste0("chr", chromosome))
cat("Number of unique human lncRNA transcripts:\n")
print(nrow(human_lncRNA_transcripts))

# Get the human PCG transcripts and setup a fixed region around TSS
human_pcg_transcripts <- data_human_selected %>%
    select(pcg_id, chromosome, pcg_tss) %>%
    mutate(region_start = as.integer(floor(pcg_tss - fix_width / 2)),
           region_end = as.integer(floor(pcg_tss + fix_width / 2))) %>%
    distinct() %>%
    select(chromosome, region_start, region_end, pcg_id) %>%
    filter(chromosome %in% c(1:22, "X")) %>%
    mutate(chromosome = paste0("chr", chromosome))
cat("Number of unique human PCG transcripts:\n")
print(nrow(human_pcg_transcripts))

# save the fixed width transcripts data
tmp_out_dir = "/mnt/c/Users/Marti/Documents/Projects/lncRNA_TF_pairs_analysis/Data/Liftover/"
updated_date = "260109"
write.table(human_lncRNA_transcripts, file = paste0(tmp_out_dir, "human_hg38_lncRNA.bed"),
            sep = "\t", quote = FALSE, row.names = FALSE, col.names = FALSE)
write.table(human_pcg_transcripts, file = paste0(tmp_out_dir, "human_hg38_PCG.bed"),
            sep = "\t", quote = FALSE, row.names = FALSE, col.names = FALSE)

Number of unique human lncRNA transcripts:
[1] 10637
Number of unique human PCG transcripts:
[1] 1915


In [ ]:
head(human_lncRNA_transcripts)